# Encode cả kho Silver bằng mô hình gom cụm (GPU)

Bước **chậm nhất** của pipeline gom cụm (`src/eventrank/clustering/pipeline.py`) là encode ~29 nghìn bài thành vector:
~45 phút trên CPU, **~2–4 phút trên GPU T4**. Notebook này chỉ làm bước đó trên Kaggle; phần gom cụm (vài giây) vẫn chạy
ở máy và tự dùng lại vector.

**Cách chạy:** Settings → Accelerator **GPU T4 x1**, **Internet on**. Add-ons → **Secrets** (bật cho notebook):
`R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET` (= `vietnam-news-datalake`) — nên dùng khoá
**chỉ đọc**, không dán khoá vào ô code. Không dùng R2: tạo Kaggle Dataset chứa thư mục `silver/<báo>/*.jsonl` và thư mục
mô hình, notebook tự tìm trong `/kaggle/input`. Run All.

**Đầu ra** (`/kaggle/working/`, tab Output → Download):
- `emb_bkai_ft_v1_s44.npz` — `ids` (article_id), `E` (N × 768, float32, đã chuẩn hoá L2), `path` = `model_version`,
  `prep`. **Đúng định dạng đệm của pipeline.**
- `encode_manifest.json` — số bài theo báo, thời gian, kiểm tra GPU ↔ CPU.

**Sau đó ở máy:** chép file `.npz` vào `data/events/cache/` (ghi đè file cùng tên) rồi `make cluster` — pipeline thấy
vector có sẵn, chỉ encode các bài ở máy mà Kaggle không có (nếu Silver ở máy mới hơn R2).

## 0. Cấu hình

Phải **khớp `configs/clustering.yaml`**: cùng mô hình, cùng `model_version`, cùng tiền xử lý (`pyvi` = NFC + tách từ —
bắt buộc với bkai). Văn bản = `tiêu đề. sapo`, làm sạch giống `pipeline.load_silver` (NFC + gộp khoảng trắng).

In [ ]:
import os
CFG = dict(
    r2_silver="silver/",                                     # silver/<báo>/*.jsonl
    r2_model="models/embedding_v1_NguyenDinhHieu/model/",   # = data/models/bkai_ft_v1_s44 ở máy
    model_version="bkai_ft_v1_s44",                          # tên file đệm + khoá đệm, phải trùng configs/clustering.yaml
    prep="pyvi",
    max_seq_length=256,
    batch_size=128,
)
SMOKE = os.environ.get("SMOKE", "0") == "1"                  # chạy thử: chỉ 500 bài
CFG

## 1. Môi trường + tải dữ liệu, mô hình

In [ ]:
import importlib.util, subprocess, sys
need = [p for p, m in [("pyvi", "pyvi"), ("boto3", "boto3")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

import glob, json, re, time, unicodedata
from collections import Counter
from pathlib import Path
import numpy as np

KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("OUT_DIR", "out_encode"))
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)


def secrets():
    keys = ["R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET"]
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        vals = {k: c.get_secret(k) for k in keys}
    except Exception:
        vals = {k: os.environ.get(k) for k in keys}       # chạy ngoài Kaggle: lấy từ biến môi trường
    return vals if all(vals.values()) else None


def r2_download(prefix, dest, suffix=None):
    import boto3
    s = secrets()
    if s is None:
        raise RuntimeError("Thiếu Kaggle Secrets R2_* — thêm secrets hoặc dùng Kaggle Dataset (xem đầu notebook)")
    s3 = boto3.client("s3", endpoint_url=f"https://{s['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com", region_name="auto",
                      aws_access_key_id=s["R2_ACCESS_KEY_ID"], aws_secret_access_key=s["R2_SECRET_ACCESS_KEY"])
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=s["R2_BUCKET"], Prefix=prefix):
        for o in page.get("Contents", []):
            rel = o["Key"][len(prefix):]
            if not rel or (suffix and not rel.endswith(suffix)):
                continue
            out = Path(dest) / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists() or out.stat().st_size != o["Size"]:
                s3.download_file(s["R2_BUCKET"], o["Key"], str(out)); n += 1
    return n


def kaggle_input_silver():
    hits = glob.glob("/kaggle/input/**/silver/*/*.jsonl", recursive=True) if KAGGLE else []
    return sorted(hits)


def kaggle_input_model():
    for d, _, fs in os.walk("/kaggle/input", followlinks=True) if KAGGLE else []:
        if "model.safetensors" in fs and "config_sentence_transformers.json" in fs and "bkai" in d.lower():
            return d
    return None


SILVER_FILES = kaggle_input_silver() or sorted(glob.glob(str(DATA / "silver/*/*.jsonl")))   # chạy lại: không tải lại
if not SILVER_FILES:
    print("tải Silver từ R2:", r2_download(CFG["r2_silver"], DATA / "silver", suffix=".jsonl"), "file")
    SILVER_FILES = sorted(glob.glob(str(DATA / "silver/*/*.jsonl")))
MODEL_DIR = kaggle_input_model()
if MODEL_DIR is None and (DATA / "model/model.safetensors").exists():
    MODEL_DIR = str(DATA / "model")
if MODEL_DIR is None:
    MODEL_DIR = str(DATA / "model")
    print("tải mô hình từ R2:", r2_download(CFG["r2_model"], MODEL_DIR), "file")
print(len(SILVER_FILES), "file Silver:", [Path(f).parent.name + "/" + Path(f).name for f in SILVER_FILES])
print("mô hình:", MODEL_DIR)

## 2. Đọc Silver → văn bản

Giống `pipeline.load_silver`: bỏ trùng `article_id` (giữ bản đầu tiên theo thứ tự file), văn bản = `tiêu đề. sapo` đã làm
sạch. Encode **mọi** bài có `article_id` (kể cả bài thiếu `published_at` — pipeline tự bỏ qua bài đó, thừa không sao).

In [ ]:
_SPACE = re.compile(r"\s+")
clean = lambda s: _SPACE.sub(" ", unicodedata.normalize("NFC", s or "")).strip()


def head_text(title, sapo):
    return f"{title}. {sapo}".strip(" .") if sapo else title


texts, src = {}, Counter()
for path in SILVER_FILES:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            a = json.loads(line)
            if a["article_id"] in texts:
                continue
            texts[a["article_id"]] = head_text(clean(a.get("title")), clean(a.get("sapo")))
            src[(a.get("source") or "").lower()] += 1
ids = sorted(texts)
if SMOKE:
    ids = ids[:500]
print(len(ids), "bài ·", dict(src))

## 3. Encode trên GPU

Tách từ `pyvi` chạy CPU (~1 phút), sau đó encode GPU. Giữ **float32** (không fp16) để vector khớp bản CPU ở máy.

In [ ]:
import torch
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("thiết bị:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
t0 = time.time()
prepped = [ViTokenizer.tokenize(unicodedata.normalize("NFC", texts[i])) for i in ids]
t_tok = time.time() - t0
model = SentenceTransformer(MODEL_DIR, device=device)
model.max_seq_length = CFG["max_seq_length"]
t1 = time.time()
E = model.encode(prepped, batch_size=CFG["batch_size"], normalize_embeddings=True, show_progress_bar=True,
                 convert_to_numpy=True).astype(np.float32)
t_enc = time.time() - t1
print(f"tách từ {t_tok:.0f}s · encode {t_enc:.0f}s · {E.shape}")

## 4. Kiểm tra

- Mọi vector có độ dài 1, không có NaN.
- **GPU ↔ CPU**: encode lại 64 bài trên CPU, cosine giữa hai bản phải ≈ 1 (≥ 0,999). Nếu thấp hơn → không dùng file này.

In [ ]:
norms = np.linalg.norm(E, axis=1)
assert np.isfinite(E).all(), "có NaN/inf"
assert np.allclose(norms, 1, atol=1e-3), (norms.min(), norms.max())
rng = np.random.default_rng(0)
pick = rng.choice(len(ids), size=min(64, len(ids)), replace=False)
cpu = SentenceTransformer(MODEL_DIR, device="cpu")
cpu.max_seq_length = CFG["max_seq_length"]
Ec = cpu.encode([prepped[k] for k in pick], batch_size=16, normalize_embeddings=True).astype(np.float32)
agree = (E[pick] * Ec).sum(1)
print(f"GPU ↔ CPU cosine: min {agree.min():.5f} · trung bình {agree.mean():.5f}")
assert agree.min() >= 0.999, "GPU lệch CPU quá nhiều — kiểm tra lại phiên bản thư viện"

## 5. Ghi đầu ra

In [ ]:
out = WORK / f"emb_{CFG['model_version']}.npz"
np.savez(out, ids=np.array(ids), E=E, path=CFG["model_version"], prep=CFG["prep"])
import sentence_transformers, transformers
manifest = dict(config=CFG, n=len(ids), sources=dict(src), smoke=SMOKE, device=device,
                seconds_tokenize=round(t_tok, 1), seconds_encode=round(t_enc, 1),
                gpu_cpu_cosine_min=float(agree.min()), gpu_cpu_cosine_mean=float(agree.mean()),
                versions=dict(torch=torch.__version__, sentence_transformers=sentence_transformers.__version__,
                              transformers=transformers.__version__))
(WORK / "encode_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
print(out, f"{out.stat().st_size / 1e6:.1f} MB")
manifest